# 상위 20개 음성 필터 조합 노이즈 제거 및 PESQ 평가

이 노트북은 외부 CSV 파일을 로드하지 않고, 사전에 도출된 상위 20개 필터 조합을 코드 내에 직접 하드코딩하여 노이즈를 제거하고 PESQ 점수와 통계를 계산·저장합니다.

## 1. 실행 환경 및 필요한 라이브러리

In [ ]:
# Make Conda DLLs available even when VS Code skips environment activation.
import os
import sys
from pathlib import Path

_conda_dll_handles = []
if sys.platform == "win32" and (Path(sys.prefix) / "conda-meta").is_dir():
    _conda_bin = Path(sys.prefix) / "Library" / "bin"
    if _conda_bin.is_dir():
        os.environ["PATH"] = str(_conda_bin) + os.pathsep + os.environ.get("PATH", "")
        _conda_dll_handles.append(os.add_dll_directory(str(_conda_bin)))

import importlib.util
import subprocess

dependencies = {
    "numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.12,<2",
    "soundfile": "soundfile>=0.12,<1", "pandas": "pandas>=2.1,<4",
    "tqdm": "tqdm>=4.66,<5", "pesq": "pesq==0.0.4",
}
missing = [package for module, package in dependencies.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing), flush=True)
    result = subprocess.run([sys.executable, "-m", "pip", "install", *missing])
    if result.returncode:
        if sys.platform == "win32" and "pesq==0.0.4" in missing:
            command = subprocess.list2cmdline([
                "conda", "install", "--prefix", sys.prefix, "-c", "conda-forge",
                "pesq=0.0.4", "pysoundfile", "pandas>=2.1,<4", "tqdm>=4.66,<5",
            ])
            print("If PESQ compilation fails on Windows, run in Anaconda Prompt:")
            print(command)
            print("After installation, restart the notebook kernel and run from the first cell.")
        raise RuntimeError("Package installation failed. Check the pip error above and the setup instructions.")
print("Python:", sys.version.split()[0])


## 2. 데이터 경로 및 평가 설정

In [ ]:
from pathlib import Path

DATA_SOURCE = "google_drive"  # "local" 또는 "google_drive"
LOCAL_DATA_PATH = Path("release/noise_reduction_challenge")
GOOGLE_DRIVE_DATA_PATH = Path("/content/drive/MyDrive/Signals_an_Systems/noise_reduction_challenge")
GOOGLE_DRIVE_MOUNT_PATH = "/content/drive"

# 평가 설정
EVAL_SPLIT = "validation"        # "train" 또는 "validation"
EVAL_MAX_SAMPLES = None          # None이면 전체 파일 사용, 정수 지정 시 해당 개수만큼 샘플링
EVAL_RANDOM_STATE = 42
EVAL_RESUME = True
EVAL_CHECKPOINT_EVERY = 1


## 3. 공통 설정 및 음성 데이터 처리

In [ ]:
from dataclasses import dataclass, asdict
import json
import platform
import importlib.metadata

import numpy as np
import pandas as pd
import soundfile as sf
from scipy import signal
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm


@dataclass(frozen=True)
class Config:
    sample_rate: int = 16000
    n_fft: int = 512
    hop: int = 128
    filter_order: int = 4
    lowpass_hz: float = 3500.0
    highpass_hz: float = 100.0
    bandstop_hz: tuple = (950.0, 1050.0)
    basic_stages: tuple = ("highpass", "lowpass", "bandstop")
    noise_fraction: float = 0.20
    subtraction_alpha: float = 1.5
    subtraction_floor: float = 0.02
    wiener_smoothing: float = 0.98
    wiener_gain_floor: float = 0.05
    bandpass_hz: tuple = (120.0, 3800.0)
    notch_hz: float = 1000.0
    notch_q: float = 30.0
    gating_threshold: float = 1.5
    gating_width: float = 1.5
    gating_gain_floor: float = 0.03
    adaptive_noise_ratio: float = 1.5
    adaptive_attack: float = 0.90
    adaptive_release: float = 0.995
    adaptive_gain_floor: float = 0.05
    subspace_threshold: float = 1.0
    subspace_max_rank: int = 48

    def __post_init__(self):
        if self.sample_rate != 16000:
            raise ValueError("This baseline uses 16000 Hz / wideband PESQ.")
        if not 0 < self.hop <= self.n_fft or self.n_fft < 4:
            raise ValueError("Require 0 < hop <= n_fft and n_fft >= 4.")
        if not signal.check_NOLA("hann", self.n_fft, self.n_fft - self.hop):
            raise ValueError("STFT window/hop does not satisfy NOLA.")
        if not 0 < self.noise_fraction <= 1:
            raise ValueError("noise_fraction must be in (0, 1].")
        if not 0 <= self.wiener_smoothing < 1:
            raise ValueError("wiener_smoothing must be in [0, 1).")
        if not 0 <= self.subtraction_floor <= 1 or self.subtraction_alpha < 0:
            raise ValueError("Invalid spectral subtraction parameters.")
        if not 0 <= self.wiener_gain_floor <= 1:
            raise ValueError("wiener_gain_floor must be in [0, 1].")
        nyquist = self.sample_rate / 2
        if not 0 < self.bandpass_hz[0] < self.bandpass_hz[1] < nyquist:
            raise ValueError("Invalid band-pass interval.")
        if not 0 < self.notch_hz < nyquist or self.notch_q <= 0:
            raise ValueError("Invalid notch parameters.")
        if self.gating_threshold < 0 or self.gating_width <= 0 or not 0 <= self.gating_gain_floor <= 1:
            raise ValueError("Invalid spectral gating parameters.")
        if not 0 < self.adaptive_attack <= 1 or not 0 < self.adaptive_release <= 1:
            raise ValueError("Invalid adaptive filter smoothing parameters.")
        if self.subspace_threshold < 0 or self.subspace_max_rank < 1:
            raise ValueError("Invalid subspace parameters.")
        if not 0 < self.highpass_hz < self.lowpass_hz < nyquist:
            raise ValueError("Require 0 < highpass < lowpass < Nyquist.")
        if not 0 < self.bandstop_hz[0] < self.bandstop_hz[1] < nyquist:
            raise ValueError("Invalid band-stop interval.")
        if self.filter_order < 1 or int(self.filter_order) != self.filter_order:
            raise ValueError("filter_order must be a positive integer.")


CFG = Config()


def waveform(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1 or x.size == 0 or not np.isfinite(x).all():
        raise ValueError("Expected a nonempty, finite mono waveform.")
    return x


In [ ]:
def resolve_local_data_path(data_path="release/noise_reduction_challenge", start=None):
    """Resolve an explicit local path from the working folder or its parents."""
    data_path = Path(data_path).expanduser()
    start = Path(start or Path.cwd()).resolve()
    candidates = ([data_path] if data_path.is_absolute() else
                  [parent / data_path for parent in (start, *start.parents)])
    if data_path == Path("release/noise_reduction_challenge"):
        for parent in (start, *start.parents):
            candidates.extend((parent, parent / "noise_reduction_challenge"))
    for candidate in candidates:
        if all((candidate / split / "samples.csv").is_file()
               for split in ("train", "validation")):
            return candidate.resolve()
    raise FileNotFoundError(
        f"LOCAL_DATA_PATH={str(data_path)!r} could not be found from {start}. "
        "Set it to the folder containing train and validation, each with samples.csv."
    )


def load_manifest(root, split):
    if split not in ("train", "validation"):
        raise ValueError("split must be train or validation")
    root = Path(root).resolve()
    df = pd.read_csv(root / split / "samples.csv")
    required = {"segment_id", "clean_path", "noisy_path", "sample_rate_hz", "num_samples",
                "noise_category", "noise_collection", "noise_code", "noise_difficulty",
                "target_snr_db", "split"}
    if not required.issubset(df.columns) or df.empty:
        raise ValueError("Empty manifest or required columns missing.")
    if df.segment_id.duplicated().any():
        raise ValueError("Duplicate segment IDs.")
    if "split" in df and not df["split"].eq(split).all():
        raise ValueError("Manifest split mismatch.")
    for column in ("clean_path", "noisy_path"):
        for relative in df[column]:
            path = (root / relative).resolve()
            if not path.is_relative_to(root) or not path.is_file():
                raise FileNotFoundError(f"Invalid/missing file: {path}")
    return df


def read_audio(root, row, kind, cfg=CFG):
    x, sr = sf.read(Path(root) / row[f"{kind}_path"], dtype="float64")
    x = waveform(x)
    if sr != cfg.sample_rate or sr != int(row["sample_rate_hz"]):
        raise ValueError(f"Sample-rate mismatch for {row['segment_id']}: {sr}")
    if x.size != int(row["num_samples"]):
        raise ValueError(f"Length mismatch for {row['segment_id']}")
    return x


def read_pair(root, row, cfg=CFG):
    return read_audio(root, row, "clean", cfg), read_audio(root, row, "noisy", cfg)


In [ ]:
if DATA_SOURCE == "local":
    DATA_ROOT = resolve_local_data_path(LOCAL_DATA_PATH)
elif DATA_SOURCE == "google_drive":
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError("google_drive 모드는 Google Colab에서 실행하세요.") from error
    drive.mount(GOOGLE_DRIVE_MOUNT_PATH)
    DATA_ROOT = GOOGLE_DRIVE_DATA_PATH.expanduser().resolve()
else:
    raise ValueError('DATA_SOURCE는 "local" 또는 "google_drive"여야 합니다.')

OUTPUT_ROOT = DATA_ROOT / "outputs"
eval_df = load_manifest(DATA_ROOT, EVAL_SPLIT)
EVAL_OUTPUT_DIR = OUTPUT_ROOT / "top20_evaluation"
print(f"데이터 경로: {DATA_ROOT}")
print(f"평가 데이터: {EVAL_SPLIT} / {len(eval_df)}개 파일")
print(f"결과 저장 경로: {EVAL_OUTPUT_DIR}")


### 3.1 Band-pass Filter

In [ ]:
def band_pass_filter(noisy, cfg=CFG):
    x = waveform(noisy)
    low, high = cfg.bandpass_hz
    sos = signal.butter(cfg.filter_order, (low, high), btype="bandpass",
                        fs=cfg.sample_rate, output="sos")
    return signal.sosfiltfilt(sos, x, padlen=min(3 * (2 * len(sos) + 1), len(x) - 1))


### 3.2 Notch Filter

In [ ]:
def notch_filter(noisy, cfg=CFG):
    x = waveform(noisy)
    b, a = signal.iirnotch(cfg.notch_hz, cfg.notch_q, fs=cfg.sample_rate)
    padlen = min(3 * max(len(a), len(b)), len(x) - 1)
    return signal.filtfilt(b, a, x, padlen=padlen)


### 3.3 Spectral Subtraction 및 STFT 공통 함수

In [ ]:
def stft_audio(x, cfg=CFG):
    x = waveform(x)
    if x.size < cfg.n_fft:
        x = np.pad(x, (0, cfg.n_fft - x.size))
    return signal.stft(x, fs=cfg.sample_rate, window="hann", nperseg=cfg.n_fft,
                       noverlap=cfg.n_fft - cfg.hop, nfft=cfg.n_fft,
                       boundary="zeros", padded=True)


def inverse_stft(z, length, cfg=CFG):
    _, y = signal.istft(z, fs=cfg.sample_rate, window="hann", nperseg=cfg.n_fft,
                        noverlap=cfg.n_fft - cfg.hop, nfft=cfg.n_fft,
                        input_onesided=True, boundary=True)
    if len(y) < length:
        raise ValueError("Inverse STFT produced insufficient samples.")
    return waveform(y[:length])


def estimate_noise_power(z, times, length, cfg=CFG):
    power = np.abs(z) ** 2
    centers = np.rint(times * cfg.sample_rate).astype(int)
    full = np.flatnonzero((centers >= cfg.n_fft // 2) &
                          (centers + cfg.n_fft // 2 <= length))
    if not len(full):
        full = np.arange(power.shape[1])
    frame_energy = power[:, full].mean(axis=0)
    count = max(1, int(np.ceil(cfg.noise_fraction * len(full))))
    selected = full[np.argsort(frame_energy, kind="stable")[:count]]
    estimate = power[:, selected].mean(axis=1)
    return np.maximum(uniform_filter1d(estimate, size=3, mode="nearest"), 1e-18)


def spectral_subtraction(noisy, cfg=CFG):
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    power = np.abs(z) ** 2
    noise = estimate_noise_power(z, times, len(x), cfg)[:, None]
    target = np.maximum(power - cfg.subtraction_alpha * noise,
                        cfg.subtraction_floor * power)
    gain = np.sqrt(np.clip(target / np.maximum(power, 1e-18), 0, 1))
    return inverse_stft(gain * z, len(x), cfg)


### 3.4 Wiener Filter

In [ ]:
def wiener_filter(noisy, cfg=CFG):
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    power = np.abs(z) ** 2
    noise = estimate_noise_power(z, times, len(x), cfg)
    posterior_snr = power / noise[:, None]
    enhanced = np.empty_like(z)
    previous_clean_power = np.zeros_like(noise)
    for frame in range(z.shape[1]):
        instantaneous = np.maximum(posterior_snr[:, frame] - 1.0, 0.0)
        if frame == 0:
            prior_snr = instantaneous
        else:
            prior_snr = (cfg.wiener_smoothing * previous_clean_power / noise
                         + (1 - cfg.wiener_smoothing) * instantaneous)
        gain = np.clip(prior_snr / (1 + prior_snr), cfg.wiener_gain_floor, 1.0)
        enhanced[:, frame] = gain * z[:, frame]
        previous_clean_power = np.abs(enhanced[:, frame]) ** 2
    return inverse_stft(enhanced, len(x), cfg)


### 3.5 MMSE-STSA Filter

In [ ]:
def mmse_stsa_filter(noisy, cfg=CFG):
    from scipy.special import i0, i1
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    power = np.abs(z) ** 2
    noise = estimate_noise_power(z, times, len(x), cfg)[:, None]
    gamma = np.maximum(power / noise, 1e-12)
    xi = np.maximum(gamma - 1.0, 0.0)
    for frame in range(1, z.shape[1]):
        xi[:, frame] = (cfg.wiener_smoothing *
                        (np.abs(z[:, frame - 1]) ** 2 / noise[:, 0]) +
                        (1.0 - cfg.wiener_smoothing) *
                        np.maximum(gamma[:, frame] - 1.0, 0.0))
    v = np.maximum(gamma * xi / (1.0 + xi), 1e-12)
    half_v = np.minimum(v / 2.0, 700.0)
    gain = (np.sqrt(np.pi * v) / (2.0 * gamma) *
            np.exp(-half_v) *
            ((1.0 + v) * i0(half_v) + v * i1(half_v)))
    gain = np.clip(np.nan_to_num(gain, nan=0.0, posinf=1.0), cfg.wiener_gain_floor, 1.0)
    return inverse_stft(gain * z, len(x), cfg)


### 3.6 MMSE-LSA Filter

In [ ]:
def mmse_lsa_filter(noisy, cfg=CFG):
    from scipy.special import exp1
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    power = np.abs(z) ** 2
    noise = estimate_noise_power(z, times, len(x), cfg)[:, None]
    gamma = np.maximum(power / noise, 1e-12)
    xi = np.maximum(gamma - 1.0, 0.0)
    for frame in range(1, z.shape[1]):
        xi[:, frame] = (cfg.wiener_smoothing *
                        (np.abs(z[:, frame - 1]) ** 2 / noise[:, 0]) +
                        (1.0 - cfg.wiener_smoothing) *
                        np.maximum(gamma[:, frame] - 1.0, 0.0))
    v = np.maximum(gamma * xi / (1.0 + xi), 1e-12)
    gain = (xi / (1.0 + xi)) * np.exp(0.5 * exp1(v))
    gain = np.clip(np.nan_to_num(gain, nan=0.0, posinf=1.0), cfg.wiener_gain_floor, 1.0)
    return inverse_stft(gain * z, len(x), cfg)


### 3.7 Spectral Gating

In [ ]:
def spectral_gating_filter(noisy, cfg=CFG):
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    magnitude = np.abs(z)
    noise_mag = np.sqrt(estimate_noise_power(z, times, len(x), cfg))[:, None]
    threshold = cfg.gating_threshold * noise_mag
    mask = np.clip((magnitude - threshold) /
                   np.maximum(cfg.gating_width * noise_mag, 1e-12), 0.0, 1.0)
    mask = uniform_filter1d(mask, size=3, axis=1, mode="nearest")
    mask = uniform_filter1d(mask, size=3, axis=0, mode="nearest")
    mask = np.clip(mask, cfg.gating_gain_floor, 1.0)
    return inverse_stft(mask * z, len(x), cfg)


### 3.8 Adaptive Filter

In [ ]:
def adaptive_filter(noisy, cfg=CFG):
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    power = np.abs(z) ** 2
    initial = estimate_noise_power(z, times, len(x), cfg)
    noise = np.empty_like(power)
    noise[:, 0] = initial
    for frame in range(1, z.shape[1]):
        prev = noise[:, frame - 1]
        below = power[:, frame] <= cfg.adaptive_noise_ratio * prev
        rate = np.where(below, cfg.adaptive_attack, cfg.adaptive_release)
        candidate = rate * prev + (1.0 - rate) * power[:, frame]
        noise[:, frame] = np.maximum(candidate, 1e-18)
    posterior = power / np.maximum(noise, 1e-18)
    gain = np.sqrt(np.maximum(posterior - 1.0, 0.0) /
                   np.maximum(posterior, 1e-12))
    gain = np.clip(gain, cfg.adaptive_gain_floor, 1.0)
    gain = uniform_filter1d(gain, size=3, axis=1, mode="nearest")
    return inverse_stft(gain * z, len(x), cfg)


### 3.9 Subspace Filter

In [ ]:
def subspace_filter(noisy, cfg=CFG):
    """Low-rank subspace denoising using the smaller Gram matrix."""
    x = waveform(noisy)
    _, times, z = stft_audio(x, cfg)
    try:
        noise = estimate_noise_power(z, times, len(x), cfg)
        noise_level = float(np.sqrt(np.mean(noise)))
        threshold = cfg.subspace_threshold * noise_level * np.sqrt(max(z.shape))
        freq_bins, frames = z.shape
        rank = min(cfg.subspace_max_rank, freq_bins, frames)

        if frames >= freq_bins:
            gram = z @ z.conj().T
            eigenvalues, u = np.linalg.eigh(gram)
            order = np.argsort(eigenvalues)[::-1][:rank]
            singular = np.sqrt(np.maximum(eigenvalues[order], 0.0))
            u = u[:, order]
            keep = singular > 1e-12
            if not np.any(keep):
                return spectral_subtraction(x, cfg)
            singular, u = singular[keep], u[:, keep]
            vh = (u.conj().T @ z) / singular[:, None]
            shrunk = np.maximum(singular - threshold, 0.0)
            enhanced = (u * shrunk[None, :]) @ vh
        else:
            gram = z.conj().T @ z
            eigenvalues, v = np.linalg.eigh(gram)
            order = np.argsort(eigenvalues)[::-1][:rank]
            singular = np.sqrt(np.maximum(eigenvalues[order], 0.0))
            v = v[:, order]
            keep = singular > 1e-12
            if not np.any(keep):
                return spectral_subtraction(x, cfg)
            singular, v = singular[keep], v[:, keep]
            u = (z @ v) / singular[None, :]
            shrunk = np.maximum(singular - threshold, 0.0)
            enhanced = (u * shrunk[None, :]) @ v.conj().T
    except np.linalg.LinAlgError:
        return spectral_subtraction(x, cfg)
    return inverse_stft(enhanced, len(x), cfg)


## 4. 9개 필터 등록

In [ ]:
FILTER_METHODS = (
    "band_pass", "notch", "spectral_subtraction", "wiener",
    "mmse_stsa", "mmse_lsa", "spectral_gating", "adaptive", "subspace",
)

FILTER_FUNCTIONS = {
    "band_pass": band_pass_filter,
    "notch": notch_filter,
    "spectral_subtraction": spectral_subtraction,
    "wiener": wiener_filter,
    "mmse_stsa": mmse_stsa_filter,
    "mmse_lsa": mmse_lsa_filter,
    "spectral_gating": spectral_gating_filter,
    "adaptive": adaptive_filter,
    "subspace": subspace_filter,
}

if set(FILTER_METHODS) != set(FILTER_FUNCTIONS):
    raise ValueError("FILTER_METHODS와 FILTER_FUNCTIONS의 필터 구성이 일치하지 않습니다.")


## 5. 상위 20개 필터 조합 정의 (CSV 로드 없이 직접 구현) 및 PESQ 통계·저장

사전에 도출된 20개의 필터 조합을 순차적으로 적용하여 노이즈를 제거하고, 각 조합의 평균·표준편차·유효 계산 수·실패 수를 계산하여 CSV 및 JSON으로 저장합니다.

In [ ]:
from time import perf_counter

TOP20_COMBINATIONS = [
    # Rank 1: band_pass+wiener+subspace
    ["band_pass", "wiener", "subspace"],
    # Rank 2: band_pass+notch+wiener+subspace
    ["band_pass", "notch", "wiener", "subspace"],
    # Rank 3: band_pass+wiener
    ["band_pass", "wiener"],
    # Rank 4: band_pass+wiener+mmse_stsa+subspace
    ["band_pass", "wiener", "mmse_stsa", "subspace"],
    # Rank 5: band_pass+notch+wiener
    ["band_pass", "notch", "wiener"],
    # Rank 6: band_pass+notch+wiener+mmse_stsa+subspace
    ["band_pass", "notch", "wiener", "mmse_stsa", "subspace"],
    # Rank 7: band_pass+mmse_stsa+mmse_lsa+subspace
    ["band_pass", "mmse_stsa", "mmse_lsa", "subspace"],
    # Rank 8: band_pass+notch+mmse_stsa+mmse_lsa+subspace
    ["band_pass", "notch", "mmse_stsa", "mmse_lsa", "subspace"],
    # Rank 9: band_pass+wiener+mmse_lsa+subspace
    ["band_pass", "wiener", "mmse_lsa", "subspace"],
    # Rank 10: band_pass+notch+wiener+mmse_lsa+subspace
    ["band_pass", "notch", "wiener", "mmse_lsa", "subspace"],
    # Rank 11: band_pass+spectral_subtraction+mmse_stsa+mmse_lsa+subspace
    ["band_pass", "spectral_subtraction", "mmse_stsa", "mmse_lsa", "subspace"],
    # Rank 12: band_pass+spectral_gating+subspace
    ["band_pass", "spectral_gating", "subspace"],
    # Rank 13: band_pass+mmse_stsa+spectral_gating+subspace
    ["band_pass", "mmse_stsa", "spectral_gating", "subspace"],
    # Rank 14: band_pass+notch+spectral_subtraction+mmse_stsa+mmse_lsa+subspace
    ["band_pass", "notch", "spectral_subtraction", "mmse_stsa", "mmse_lsa", "subspace"],
    # Rank 15: band_pass+notch+spectral_gating+subspace
    ["band_pass", "notch", "spectral_gating", "subspace"],
    # Rank 16: band_pass+notch+mmse_stsa+spectral_gating+subspace
    ["band_pass", "notch", "mmse_stsa", "spectral_gating", "subspace"],
    # Rank 17: band_pass+mmse_lsa+spectral_gating+subspace
    ["band_pass", "mmse_lsa", "spectral_gating", "subspace"],
    # Rank 18: band_pass+spectral_subtraction+mmse_lsa+subspace
    ["band_pass", "spectral_subtraction", "mmse_lsa", "subspace"],
    # Rank 19: band_pass+mmse_stsa+mmse_lsa
    ["band_pass", "mmse_stsa", "mmse_lsa"],
    # Rank 20: band_pass+spectral_subtraction+mmse_stsa+subspace
    ["band_pass", "spectral_subtraction", "mmse_stsa", "subspace"],
]

def pesq_score(clean, estimate, sample_rate=16000):
    """PESQ-WB 점수를 계산합니다. 반환값: (점수, 오류 메시지)"""
    from pesq import pesq, PesqError
    try:
        value = float(pesq(
            sample_rate,
            waveform(clean),
            waveform(estimate),
            mode="wb",
            on_error=PesqError.RAISE_EXCEPTION,
        ))
        if not np.isfinite(value) or value < 0:
            raise ValueError(f"Invalid PESQ return value: {value}")
        return value, ""
    except (PesqError, ValueError) as error:
        return np.nan, f"{type(error).__name__}: {error}"


def apply_filter_combination(noisy, selected_methods, cfg=CFG):
    """선택된 필터 리스트 순서대로 노이즈 제거 필터를 연속 적용합니다."""
    enhanced = waveform(noisy).copy()

    for method in selected_methods:
        enhanced = waveform(FILTER_FUNCTIONS[method](enhanced, cfg))

    if enhanced.shape != np.asarray(noisy).shape:
        raise ValueError("필터 조합의 출력 길이가 입력과 다릅니다.")

    return enhanced


def evaluate_top20_combinations(
    root,
    manifest,
    combinations_list=TOP20_COMBINATIONS,
    split="validation",
    max_samples=None,
    random_state=42,
    output_dir=None,
    cfg=CFG,
    resume=True,
):
    """
    코드에 직접 정의된 20개 필터 조합을 순회하며 노이즈를 제거하고 PESQ-WB 통계를 계산합니다.
    """
    import pesq

    if split not in ("train", "validation"):
        raise ValueError("split은 'train' 또는 'validation'이어야 합니다.")
    if manifest.empty:
        raise ValueError("평가에 사용할 manifest가 비어 있습니다.")
    if "split" in manifest.columns and not manifest["split"].eq(split).all():
        raise ValueError(f"전달된 manifest는 {split} split이어야 합니다.")
    if max_samples is not None and max_samples < 1:
        raise ValueError("max_samples는 양의 정수 또는 None이어야 합니다.")

    # 샘플 추출
    if max_samples is not None and len(manifest) > max_samples:
        eval_manifest = (
            manifest.sample(n=max_samples, random_state=random_state)
            .sort_index()
            .reset_index(drop=True)
        )
    else:
        eval_manifest = manifest.reset_index(drop=True).copy()

    # 데이터 로드 및 캐싱
    cached_samples = []
    print(f"평가 데이터 준비: {len(eval_manifest)}개 파일 ({split})")
    for _, row in tqdm(
        eval_manifest.iterrows(),
        total=len(eval_manifest),
        desc="Load evaluation samples",
    ):
        clean, noisy = read_pair(root, row, cfg)
        valid = int(row.get("valid_samples", len(clean)))
        if not 0 < valid <= len(clean) or valid > len(noisy):
            raise ValueError(f"잘못된 valid_samples: {row['segment_id']}")

        cached_samples.append({
            "segment_id": row["segment_id"],
            "clean": clean,
            "noisy": noisy,
            "valid": valid,
        })

    # PESQ 사전 검사 (원본 noisy 기준)
    preflight_errors = []
    preflight_successes = 0
    for sample in cached_samples[:5]:
        try:
            baseline_score, baseline_error = pesq_score(
                sample["clean"][:sample["valid"]],
                sample["noisy"][:sample["valid"]],
                cfg.sample_rate,
            )
            if np.isfinite(baseline_score):
                preflight_successes += 1
            else:
                preflight_errors.append(
                    f"{sample['segment_id']}: {baseline_error or 'PESQ returned NaN'}"
                )
        except Exception as exc:
            preflight_errors.append(
                f"{sample['segment_id']}: {type(exc).__name__}: {exc}"
            )

    if preflight_successes == 0:
        examples = "\n".join(preflight_errors[:5]) or "오류 메시지가 반환되지 않았습니다."
        raise RuntimeError(
            "PESQ 사전 검사가 실패했습니다. 원본 noisy 음성의 PESQ 계산부터 실패한 상태입니다.\n"
            f"샘플레이트: {cfg.sample_rate} Hz / 검사 파일: {min(5, len(cached_samples))}개\n"
            f"오류 예시:\n{examples}"
        )

    print(f"PESQ 사전 검사 통과: {preflight_successes}/{min(5, len(cached_samples))}개")

    # 출력 폴더 및 체크포인트 준비
    if output_dir is None:
        output_dir = Path(root) / "outputs" / "top20_evaluation"
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    sample_tag = f"n{max_samples}" if max_samples is not None else "all"
    checkpoint_path = output_dir / f"{split}_top20_checkpoint_{sample_tag}.csv"

    if resume and checkpoint_path.exists():
        previous = pd.read_csv(checkpoint_path)
        result_rows = previous.to_dict("records")
        completed_combos = set(previous["combination"].astype(str))
        print(f"체크포인트 불러오기: {len(completed_combos)}/{len(combinations_list)}개 조합 완료")
    else:
        result_rows = []
        completed_combos = set()

    pending_combinations = [
        methods for methods in combinations_list
        if ("+".join(methods) if methods else "(no_filter)") not in completed_combos
    ]

    started = perf_counter()
    print(f"이번 실행에서 평가할 조합: {len(pending_combinations)}개")

    combo_bar = tqdm(
        pending_combinations,
        desc="Top 20 조합 평가",
        position=0,
        mininterval=0.5,
    )
    for selected_methods in combo_bar:
        combo_label = "+".join(selected_methods) if selected_methods else "(no_filter)"
        combo_started = perf_counter()

        scores = []
        errors = 0
        error_examples = []

        sample_bar = tqdm(
            cached_samples,
            desc=f"샘플 처리: {combo_label[:32]}",
            position=1,
            leave=False,
            mininterval=0.5,
        )
        for sample in sample_bar:
            sample_started = perf_counter()
            try:
                enhanced = apply_filter_combination(
                    sample["noisy"], selected_methods, cfg
                )
                enhanced_valid = waveform(enhanced[:sample["valid"]])

                score, error = pesq_score(
                    sample["clean"][:sample["valid"]],
                    enhanced_valid,
                    cfg.sample_rate,
                )

                if np.isfinite(score):
                    scores.append(score)
                else:
                    errors += 1
                    if error and len(error_examples) < 3:
                        error_examples.append(str(error))

            except Exception as exc:
                errors += 1
                if len(error_examples) < 3:
                    error_examples.append(f"{type(exc).__name__}: {exc}")
            finally:
                sample_bar.set_postfix(
                    file=str(sample["segment_id"])[-22:],
                    sec=f"{perf_counter() - sample_started:.1f}",
                    refresh=False,
                )

        combo_elapsed = perf_counter() - combo_started
        combo_bar.set_postfix(
            current=combo_label[:24],
            combo_sec=f"{combo_elapsed:.1f}",
            valid=len(scores),
            failed=errors,
            refresh=False,
        )

        # 조합에 해당하는 9자리 비트 마스크 생성
        bits = "".join("1" if m in selected_methods else "0" for m in FILTER_METHODS)

        result_rows.append({
            "combination": combo_label,
            "filter_count": len(selected_methods),
            "filters": json.dumps(selected_methods, ensure_ascii=False),
            "pesq_mean": float(np.mean(scores)) if scores else np.nan,
            "pesq_std": float(np.std(scores, ddof=1)) if len(scores) > 1 else np.nan,
            "pesq_valid": len(scores),
            "pesq_failed": errors,
            "pesq_error_examples": " | ".join(error_examples),
            "sample_count": len(cached_samples),
            "combination_bits": bits,
        })

        if len(result_rows) % max(1, EVAL_CHECKPOINT_EVERY) == 0:
            pd.DataFrame(result_rows).to_csv(
                checkpoint_path, index=False, encoding="utf-8-sig"
            )

    pd.DataFrame(result_rows).to_csv(
        checkpoint_path, index=False, encoding="utf-8-sig"
    )
    results = pd.DataFrame(result_rows)
    if results["pesq_valid"].sum() == 0:
        raise RuntimeError(
            "20개 조합 모두 PESQ 계산에 실패했습니다. pesq_error_examples를 확인하세요."
        )

    results = results.sort_values(
        ["pesq_mean", "pesq_valid"],
        ascending=[False, False],
        na_position="last",
    ).reset_index(drop=True)
    results.insert(0, "rank", np.arange(1, len(results) + 1))

    # 최종 결과 저장
    out_csv_path = output_dir / f"{split}_top20_evaluated_{sample_tag}.csv"
    results.to_csv(out_csv_path, index=False, encoding="utf-8-sig")

    elapsed = perf_counter() - started
    metadata = {
        "split": split,
        "sample_count": len(cached_samples),
        "max_samples": max_samples,
        "random_state": random_state,
        "filter_order": list(FILTER_METHODS),
        "total_combinations": len(combinations_list),
        "metric": "PESQ-WB",
        "ranking": "descending mean PESQ",
        "elapsed_seconds": elapsed,
    }
    (output_dir / f"{split}_top20_eval_config.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"\n상위 20개 조합 평가 완료: {len(results)}개 조합")
    print(f"평가 파일 수: {len(cached_samples)}")
    print(f"소요 시간: {elapsed:.1f}초")
    print("저장된 결과 CSV:", out_csv_path)

    print("\n평가 결과 요약:")
    print(results.round(4).to_string(index=False))
    return results


# 실행 셀
eval_results = evaluate_top20_combinations(
    root=DATA_ROOT,
    manifest=eval_df,
    combinations_list=TOP20_COMBINATIONS,
    split=EVAL_SPLIT,
    max_samples=EVAL_MAX_SAMPLES,
    random_state=EVAL_RANDOM_STATE,
    output_dir=EVAL_OUTPUT_DIR,
    cfg=CFG,
    resume=EVAL_RESUME,
)

# 최고 성능 조합 출력
best_result = eval_results.iloc[0]
print("\n최고 PESQ 조합")
print("순위:", int(best_result["rank"]))
print("필터:", best_result["combination"])
print("평균 PESQ:", round(float(best_result["pesq_mean"]), 4))
print("표준편차 PESQ:", round(float(best_result["pesq_std"]), 4) if pd.notna(best_result["pesq_std"]) else "N/A")
